# Complement Colab Training

This notebook runs the repository training pipeline on Google Colab. It is designed to handle disconnects by saving results and checkpoints to Google Drive.

Before running the full grid, use the smoke run first. For the full experiment, set `RUN_FULL_GRID = True` in the launch cell near the bottom.

## 1. Select A GPU Runtime

In Colab, choose `Runtime -> Change runtime type -> GPU` before running the notebook.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, cwd=cwd, check=True)


## 2. Mount Google Drive

Checkpoints, metrics, configs, and final models will be saved in Drive so rerunning this notebook can resume after a Colab disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/complement-results")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(DRIVE_RESULTS_DIR)


## 3. Clone Or Update The Repository

Set `REPO_URL` to your GitHub repository. Public HTTPS repos work directly. For private repos, use Colab secrets or an authenticated clone method rather than writing tokens into the notebook.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/complement.git"
REPO_DIR = Path("/content/complement")

if REPO_DIR.exists():
    run("git pull", cwd=REPO_DIR)
else:
    run(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print(Path.cwd())


## 4. Install The Project

In [ ]:
run("python -m pip install -e .", cwd=REPO_DIR)


## 5. Check GPU

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available. In Colab, select Runtime -> Change runtime type -> GPU.")


## 6. Write Colab Configs

The important part for disconnect recovery is `experiment.results_dir`: it points to Google Drive. The training code writes `checkpoint.pt` after batches/epochs and resumes from the same deterministic run directories.

In [ ]:
import yaml

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

def save_yaml(path, config):
    with open(path, "w", encoding="utf-8") as file:
        yaml.safe_dump(config, file, sort_keys=False)

smoke_config = load_yaml("configs/smoke.yaml")
smoke_config["experiment"]["results_dir"] = str(DRIVE_RESULTS_DIR)
smoke_config["training"]["device"] = "cuda"
save_yaml("configs/colab_smoke.yaml", smoke_config)

full_config = load_yaml("configs/default.yaml")
full_config["experiment"]["results_dir"] = str(DRIVE_RESULTS_DIR)
full_config["training"]["device"] = "cuda"
save_yaml("configs/colab_default.yaml", full_config)

print("Wrote configs/colab_smoke.yaml")
print("Wrote configs/colab_default.yaml")


## 7. Run A Smoke Test

This should finish quickly and prove that the model, dataset, losses, logging, checkpointing, and saving all work on the Colab GPU.

In [ ]:
run("python scripts/run_grid.py --config configs/colab_smoke.yaml", cwd=REPO_DIR)


## 8. Launch Or Resume The Full Grid

Set `RUN_FULL_GRID = True` only when you are ready. If Colab disconnects, reconnect, rerun the setup cells, and run this cell again. Existing completed coordinates will be skipped/resumed from checkpoints in Drive.

In [ ]:
RUN_FULL_GRID = False

if RUN_FULL_GRID:
    run("python scripts/run_grid.py --config configs/colab_default.yaml", cwd=REPO_DIR)
else:
    print("Full grid not launched. Set RUN_FULL_GRID = True when ready.")


## 9. Inspect Saved Results

In [ ]:
run(f"find {DRIVE_RESULTS_DIR} -maxdepth 5 -type f | head -50")
